# Pipeline OCR → HTML com OpenRouter
OCR com PaddleOCR e HTML gerado por LLM na OpenRouter, com métricas por etapa, tudo de `pipeline.py`. A chave vem de `OPENROUTER_API_KEY` ou do `.env`.

**No Colab:** guarde `OPENROUTER_API_KEY` em Secrets (🔑) e envie o `colab.zip` quando pedido.

In [1]:
MODEL = "qwen/qwen3.8-27b:free"
IMAGE = "input/amostra_diversa/captura/300/01-1-cinza-300/p01.png"
KINDS = ("compact",)  # entradas do llm: compact, markdown, full_json
OUTPUT = "output/notebook"
CPU_THREADS = 2
REASONING = False  # ligado, o qwen3.8 gasta 5–16 mil tokens pensando e pode truncar

In [2]:
import os, sys
from pathlib import Path

if "google.colab" in sys.modules:
    ROOT = Path("/content/projeto")
    if not (ROOT / "pipeline.py").is_file():
        import zipfile
        from google.colab import files
        zipfile.ZipFile(next(iter(files.upload()))).extractall(ROOT)
    from google.colab import userdata
    os.environ["OPENROUTER_API_KEY"] = userdata.get("OPENROUTER_API_KEY")
    !pip install -q paddleocr==3.7.0 paddlepaddle==3.3.1 "paddlex[ocr]==3.7.2" openai python-dotenv beautifulsoup4 opencv-contrib-python
else:
    ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))
os.environ.setdefault("PADDLE_PDX_DISABLE_MODEL_SOURCE_CHECK", "True")

from IPython.display import IFrame, display
import pipeline as p

image, output = ROOT / IMAGE, ROOT / OUTPUT
show = lambda prefix: display(p.summary().filter(like=prefix, axis=0).round(2))

## 1. OCR

In [3]:
with p.stage("ocr:carregar_modelo"):
    engine = p.load_engine(cpu_threads=CPU_THREADS)
folder, status = p.process_file(engine, image, image.parent, output)
layout = next(folder.glob("*_layout.json"))
print(status, layout)
show("ocr")

/home/larissa/Downloads/InvoiceConverterPoC-NLP-26.2/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


which: no ccache in (/home/larissa/.config/Code - OSS/User/globalStorage/github.copilot-chat/debugCommand:/home/larissa/.config/Code - OSS/User/globalStorage/github.copilot-chat/copilotCli:/home/larissa/.local/bin:/home/larissa/.config/Code - OSS/User/globalStorage/github.copilot-chat/debugCommand:/home/larissa/.config/Code - OSS/User/globalStorage/github.copilot-chat/copilotCli:/usr/local/bin:/usr/bin:/usr/local/sbin:/usr/bin/site_perl:/usr/bin/vendor_perl:/usr/bin/core_perl)
/home/larissa/Downloads/InvoiceConverterPoC-NLP-26.2/.venv/lib/python3.12/site-packages/paddle/utils/cpp_extension/extension_utils.py:712: UserWarning: No ccache found. Please be aware that recompiling all source files may be required. You can download and install ccache from: https://github.com/ccache/ccache/blob/master/doc/INSTALL.md
  warnings.warn(warning_message)


Creating model: ('PP-LCNet_x1_0_doc_ori', None, None)


Model files already exist. Using cached files. To redownload, please delete the directory manually: `/home/larissa/.paddlex/official_models/PP-LCNet_x1_0_doc_ori`.


Creating model: ('PP-DocBlockLayout', None, None)


...

Creating model: ('latin_PP-OCRv5_mobile_rec', None, None)


Model files already exist. Using cached files. To redownload, please delete the directory manually: `/home/larissa/.paddlex/official_models/latin_PP-OCRv5_mobile_rec`.


/home/larissa/Downloads/InvoiceConverterPoC-NLP-26.2/pipeline.py:223: RuntimeWarning: overflow encountered in scalar multiply
  return max(0, min(x2, c) - max(x1, a)) * max(0, min(y2, d) - max(y1, b)) / max(1, (x2-x1)*(y2-y1))


ok /home/larissa/Downloads/InvoiceConverterPoC-NLP-26.2/output/intermediarios/notebook/p01_png/p01_page_001_layout.json


,calls,seconds,cpu_%,peak_ram_mb,peak_vram_mb,tokens_in,tokens_out,cost_usd
stage,,,,,,,,
ocr:carregar_modelo,1,10.24,106.41,1548.75,NaN,0,0,0.0
ocr:preparar_imagem,1,0.02,35.95,1548.72,NaN,0,0,0.0
ocr:predict,1,41.79,191.53,1832.78,NaN,0,0,0.0
ocr:reparar_tabelas,1,0.14,176.38,1841.45,NaN,0,0,0.0
ocr:salvar,1,0.43,77.55,1846.21,NaN,0,0,0.0


## 2. HTML via OpenRouter

In [4]:
logs = [p.write_html(layout, kind, output, engine="openrouter", model=MODEL, reasoning=REASONING)
        for kind in KINDS]
display(logs)
show("html")

HTML (compact): /home/larissa/Downloads/InvoiceConverterPoC-NLP-26.2/output/html/notebook/p01_png/p01_page_001_html.html


[{'source': '/home/larissa/Downloads/InvoiceConverterPoC-NLP-26.2/output/intermediarios/notebook/p01_png/p01_page_001_layout.json',
  'engine': 'openrouter',
  'status': 'accepted',
  'model': 'qwen/qwen3.8-27b:free',
  'tokens_in': 2165,
  'tokens_out': 5084,
  'cost_usd': 0.0,
  'duration_seconds': 41.682}]

,calls,seconds,cpu_%,peak_ram_mb,peak_vram_mb,tokens_in,tokens_out,cost_usd
stage,,,,,,,,
html:openrouter:compact,1,41.68,2.58,1847.76,NaN,2165,5084,0.0


In [5]:
IFrame(os.path.relpath(p.output_path(layout, KINDS[0], output)), width="100%", height=600)

## 3. Resumo

In [6]:
p.report("notebook");

                         calls  seconds     cpu_%  peak_ram_mb peak_vram_mb  tokens_in  tokens_out  cost_usd
stage                                                                                                       
ocr:carregar_modelo          1  10.2423  106.4060    1548.7461          NaN          0           0       0.0
ocr:preparar_imagem          1   0.0200   35.9451    1548.7227          NaN          0           0       0.0
ocr:predict                  1  41.7886  191.5349    1832.7773          NaN          0           0       0.0
ocr:reparar_tabelas          1   0.1437  176.3783    1841.4453          NaN          0           0       0.0
ocr:salvar                   1   0.4339   77.5463    1846.2148          NaN          0           0       0.0
html:openrouter:compact      1  41.6823    2.5786    1847.7617          NaN       2165        5084       0.0
Métricas: /home/larissa/Downloads/InvoiceConverterPoC-NLP-26.2/runs/metrics_20261003_173144_396945.json


In [7]:
if "google.colab" in sys.modules:  # baixa output/ e runs/ num zip
    !cd {ROOT} && zip -qr /content/colab_resultados.zip output runs
    from google.colab import files
    files.download("/content/colab_resultados.zip")